# 05 - Treat Residential Care

## Input
- `data/raw/admission/` — residential files + combined files 2019–2024
- `data/raw/service_users_CURF/` — individual-level CURF snapshots 2018–2024

## Output
- `data/clean/residential_admissions_by_acpr.csv` — one row per ACPR × year
- `data/clean/residential_users_by_acpr.csv` — one row per ACPR × year

## Columns
- `year`, `state`, `acpr_code`, `acpr_name`
- `n_permanent`, `n_respite`
- `n_first_admission`, `n_repeat_admission` (admissions only)
- `n_age_<group>`
- `n_male`, `n_female`
- `n_indigenous`, `n_nesb`, `n_english_speaking`

## Note
- Keeps residential care only (permanent + respite), drops all other care types
- `n_permanent`/`n_respite` derived from `care_type` or `admission_type` depending on file year

In [ ]:
import pandas as pd
import numpy as np
import os
import re

RAW_ADM           = '../../data/raw/admission'
RAW_CURF          = '../../data/raw/service_users_CURF'
SNAP_DIR          = '../../data/raw/service_users_snapshot_SA3'
OUT_RESID_ADM     = '../../data/clean/residential_admissions_by_acpr.csv'
OUT_RESID_USERS   = '../../data/clean/residential_users_by_acpr.csv'

GROUP_COLS = ['acpr_code', 'acpr_name', 'state', 'year']

WANTED = {
    'ACPR_code', 'ACPR_CODE', 'ACPR_CODE_2018',
    'ACPR_name', 'ACPR_NAME', 'ACPR_NAME_2018',
    'State', 'STATE',
    'Year', 'YEAR', 'Financial_year', 'FINANCIAL_YEAR',
    'Care_type', 'CARE_TYPE',
    'Admission_type', 'ADMISSION_TYPE',
    'First_admission', 'First _admission', 'FIRST_ADMISSION',
    'Age_group', 'AGE_GROUP', 'AGE_GROUP_5',
    'Sex', 'SEX',
    'Indigenous_status', 'INDIGENOUS_STATUS',
    'Country_of_birth', 'COUNTRY_OF_BIRTH',
    'count',
}

COL_MAP = {
    'ACPR_code': 'acpr_code', 'ACPR_CODE': 'acpr_code', 'ACPR_CODE_2018': 'acpr_code',
    'ACPR_name': 'acpr_name', 'ACPR_NAME': 'acpr_name', 'ACPR_NAME_2018': 'acpr_name',
    'State': 'state', 'STATE': 'state',
    'Year': 'year', 'YEAR': 'year', 'Financial_year': 'year', 'FINANCIAL_YEAR': 'year',
    'Care_type': 'care_type', 'CARE_TYPE': 'care_type',
    'Admission_type': 'admission_type', 'ADMISSION_TYPE': 'admission_type',
    'First_admission': 'first_admission', 'First _admission': 'first_admission', 'FIRST_ADMISSION': 'first_admission',
    'Age_group': 'age_group', 'AGE_GROUP': 'age_group', 'AGE_GROUP_5': 'age_group',
    'Sex': 'sex', 'SEX': 'sex',
    'Indigenous_status': 'indigenous_status', 'INDIGENOUS_STATUS': 'indigenous_status',
    'Country_of_birth': 'country_of_birth', 'COUNTRY_OF_BIRTH': 'country_of_birth',
}

# Residential care type values across all files
RESID_TYPES = {
    'permanent residential care', 'respite residential care',
    'residential care', 'permanent', 'respite',
}

# Maps care_type or admission_type (lowercased) to output column name
ADMT_MAP = {
    'perm': 'n_permanent', 'p': 'n_permanent',
    'permanent': 'n_permanent', 'permanent residential care': 'n_permanent',
    'resp': 'n_respite',   'r': 'n_respite',
    'respite': 'n_respite', 'respite residential care': 'n_respite',
}
ADMT_COLS = ['n_permanent', 'n_respite']

AGE_MAP = {
    '0-49': 'n_age_0_49',   '0\u201349': 'n_age_0_49',   '0049': 'n_age_0_49',
    '50-54': 'n_age_50_54', '50\u201354': 'n_age_50_54', '5054': 'n_age_50_54',
    '55-59': 'n_age_55_59', '55\u201359': 'n_age_55_59', '5559': 'n_age_55_59',
    '60-64': 'n_age_60_64', '60\u201364': 'n_age_60_64', '6064': 'n_age_60_64',
    '65-69': 'n_age_65_69', '65\u201369': 'n_age_65_69', '6569': 'n_age_65_69',
    '70-74': 'n_age_70_74', '70\u201374': 'n_age_70_74', '7074': 'n_age_70_74',
    '75-79': 'n_age_75_79', '75\u201379': 'n_age_75_79', '7579': 'n_age_75_79',
    '80-84': 'n_age_80_84', '80\u201384': 'n_age_80_84', '8084': 'n_age_80_84',
    '85-89': 'n_age_85_89', '85\u201389': 'n_age_85_89', '8589': 'n_age_85_89',
    '90-94': 'n_age_90_94', '90\u201394': 'n_age_90_94', '9094': 'n_age_90_94',
    '95-99': 'n_age_95_99', '95\u201399': 'n_age_95_99', '9599': 'n_age_95_99',
    '100+': 'n_age_100_plus',
}
AGE_COLS = [
    'n_age_0_49', 'n_age_50_54', 'n_age_55_59', 'n_age_60_64',
    'n_age_65_69', 'n_age_70_74', 'n_age_75_79', 'n_age_80_84',
    'n_age_85_89', 'n_age_90_94', 'n_age_95_99', 'n_age_100_plus',
]

In [ ]:
def load_file(path):
    if path.endswith('.xlsx'):
        try:
            return pd.read_excel(path, engine='calamine')
        except Exception:
            return pd.read_excel(path)
    try:
        return pd.read_csv(path, encoding='utf-8', low_memory=False)
    except UnicodeDecodeError:
        return pd.read_csv(path, encoding='latin1', low_memory=False)


def resolve_year(fname):
    # Use the END year of the financial year from the filename
    # e.g. '2021-22' or '2021\u201322' -> 2022
    m = re.search(r'20\d{2}[-\u2013](\d{2})', fname)
    return int('20' + m.group(1)) if m else np.nan


def get_admt_series(df):
    """Return a normalised series for perm/respite determination."""
    # Prefer admission_type column if it exists (2019-20 file)
    if 'admission_type' in df.columns:
        return df['admission_type'].astype(str).str.strip().str.lower()
    if 'care_type' in df.columns:
        return df['care_type'].astype(str).str.strip().str.lower()
    return pd.Series('', index=df.index)

---

## Part 1 — Residential Admissions (admission/)

### Input
- `data/raw/admission/` — residential-specific files (2019–2022) + combined files (2022–2024)

### Output
- `data/clean/residential_admissions_by_acpr.csv`

In [ ]:
# =============================================================================
# STEP 1 — Process residential admissions
# =============================================================================

def aggregate_admissions(df, fname):
    # 1. Slim and rename
    keep = [c for c in df.columns if c in WANTED]
    df = df[keep].copy()
    df = df.rename(columns={k: v for k, v in COL_MAP.items() if k in df.columns})

    if 'acpr_code' not in df.columns:
        print(f'  SKIP {fname} — no acpr_code')
        return None

    # 2. Year from filename, clean acpr_code
    df['year'] = resolve_year(fname)
    df['acpr_code'] = df['acpr_code'].astype(str).str.strip()
    df = df[df['acpr_code'].str.len() > 0].dropna(subset=['acpr_code'])
    if df.empty:
        return None
    df['year'] = int(df['year'].iloc[0])

    # 3. Keep residential care only
    if 'care_type' in df.columns:
        mask = df['care_type'].astype(str).str.strip().str.lower().isin(RESID_TYPES)
        df = df[mask]
    if df.empty:
        print(f'  SKIP {fname} — no residential rows')
        return None

    gcols = [c for c in GROUP_COLS if c in df.columns]

    # 4. First / repeat admission flags
    fa = df['first_admission'].astype(str).str.strip().str.lower() if 'first_admission' in df.columns else pd.Series('', index=df.index)
    df['_first']  = (fa == 'yes').astype(int)
    df['_repeat'] = (fa == 'no').astype(int)

    # 5. Permanent / respite flags
    admt = get_admt_series(df)
    admt_mapped = admt.map(ADMT_MAP)
    df['_permanent'] = (admt_mapped == 'n_permanent').astype(int)
    df['_respite']   = (admt_mapped == 'n_respite').astype(int)

    # 6. Demographic flags
    ind = df['indigenous_status'].astype(str).str.strip().str.lower() if 'indigenous_status' in df.columns else pd.Series('', index=df.index)
    df['_indigenous'] = (ind == 'indigenous').astype(int)

    sex = df['sex'].astype(str).str.strip().str.lower() if 'sex' in df.columns else pd.Series('', index=df.index)
    df['_male']   = sex.isin(['male', 'm', '1']).astype(int)
    df['_female'] = sex.isin(['female', 'f', '2']).astype(int)

    cob = df['country_of_birth'].astype(str).str.strip().str.lower() if 'country_of_birth' in df.columns else pd.Series('', index=df.index)
    df['_nesb']    = cob.str.contains('non-english', na=False).astype(int)
    df['_english'] = (cob.str.contains('english', na=False) & ~cob.str.contains('non-english', na=False)).astype(int)

    # 7. Age bracket flags
    if 'age_group' in df.columns:
        age_norm = df['age_group'].astype(str).str.strip().map(AGE_MAP)
        for col in AGE_COLS:
            df[f'_{col}'] = (age_norm == col).astype(int)
    else:
        for col in AGE_COLS:
            df[f'_{col}'] = 0

    # 8. Aggregate
    agg_map = {
        'n_first_admission':  ('_first',      'sum'),
        'n_repeat_admission': ('_repeat',     'sum'),
        'n_permanent':        ('_permanent',  'sum'),
        'n_respite':          ('_respite',    'sum'),
        'n_indigenous':       ('_indigenous', 'sum'),
        'n_male':             ('_male',       'sum'),
        'n_female':           ('_female',     'sum'),
        'n_nesb':             ('_nesb',       'sum'),
        'n_english_speaking': ('_english',    'sum'),
    }
    for col in AGE_COLS:
        agg_map[col] = (f'_{col}', 'sum')

    return df.groupby(gcols).agg(**agg_map).reset_index()


results_adm = []
for fname in sorted(os.listdir(RAW_ADM)):
    if fname.startswith('~'): continue
    if not (fname.endswith('.xlsx') or fname.endswith('.csv')): continue

    path = f'{RAW_ADM}/{fname}'
    try:
        df  = load_file(path)
        agg = aggregate_admissions(df, fname)
        if agg is not None:
            results_adm.append(agg)
            print(f'OK  {fname}: {df.shape} -> {agg.shape}  year={agg["year"].iloc[0]}')
        del df
    except Exception as e:
        print(f'ERR {fname}: {e}')

admissions = pd.concat(results_adm, ignore_index=True)
print(f'\nCombined: {admissions.shape}')
print('Years:', sorted(admissions['year'].unique()))
print('ACPRs:', admissions['acpr_code'].nunique())

In [ ]:
# =============================================================================
# STEP 2 — Quick sanity check
# =============================================================================

print(admissions[['acpr_name', 'year',
                   'n_first_admission', 'n_repeat_admission',
                   'n_permanent', 'n_respite',
                   'n_indigenous', 'n_male', 'n_female']].head(10).to_string(index=False))
print()
print('Age cols sample:')
print(admissions[['acpr_name', 'year'] + AGE_COLS].head(5).to_string(index=False))

In [ ]:
# =============================================================================
# STEP 3 — Save
# =============================================================================

admissions.to_csv(OUT_RESID_ADM, index=False)
print(f'Saved: {OUT_RESID_ADM}')
print(f'Shape: {admissions.shape}')

---

## Part 2 — Residential Users (service_users_CURF/)

### Input
- `data/raw/service_users_CURF/` — CURF snapshots 2018–2024

### Output
- `data/clean/residential_users_by_acpr.csv`

### Note
- 2018–2019 file is GEN data (pre-aggregated, has `count` column); 2020–2024 are individual-level CURFs
- `n_permanent`/`n_respite` derived directly from `care_type` column

In [ ]:
# =============================================================================
# STEP 4 — Process residential users (service_users_CURF)
# =============================================================================

def aggregate_curf_resid(df, fname):
    # 1. Slim and rename
    keep = [c for c in df.columns if c in WANTED]
    df = df[keep].copy()
    df = df.rename(columns={k: v for k, v in COL_MAP.items() if k in df.columns})

    if 'acpr_code' not in df.columns:
        print(f'  SKIP {fname} — no acpr_code')
        return None

    # 2. Clean acpr_code and year
    df['acpr_code'] = df['acpr_code'].astype(str).str.strip()
    df = df[df['acpr_code'].str.len() > 0].dropna(subset=['acpr_code'])
    df['year'] = pd.to_numeric(df['year'], errors='coerce')
    df = df.dropna(subset=['year'])
    df['year'] = df['year'].astype(int)

    # 3. Keep residential care only
    if 'care_type' in df.columns:
        mask = df['care_type'].astype(str).str.strip().str.lower().isin(RESID_TYPES)
        df = df[mask]
    if df.empty:
        print(f'  SKIP {fname} — no residential rows')
        return None

    # 4. Row weight
    if 'count' in df.columns:
        df['_weight'] = pd.to_numeric(df['count'], errors='coerce').fillna(0).astype(int)
    else:
        df['_weight'] = 1

    # 5. Permanent / respite flags
    admt = get_admt_series(df)
    admt_mapped = admt.map(ADMT_MAP)
    df['_permanent'] = (admt_mapped == 'n_permanent').astype(int) * df['_weight']
    df['_respite']   = (admt_mapped == 'n_respite').astype(int)   * df['_weight']

    # 6. Demographic flags
    ind = df['indigenous_status'].astype(str).str.strip().str.lower() if 'indigenous_status' in df.columns else pd.Series('', index=df.index)
    df['_indigenous'] = ind.isin(['indigenous', '1', '2', '3']).astype(int) * df['_weight']

    sex = df['sex'].astype(str).str.strip().str.lower() if 'sex' in df.columns else pd.Series('', index=df.index)
    df['_male']   = sex.isin(['male', 'males', '1']).astype(int)    * df['_weight']
    df['_female'] = sex.isin(['female', 'females', '2']).astype(int) * df['_weight']

    cob = df['country_of_birth'].astype(str).str.strip().str.lower() if 'country_of_birth' in df.columns else pd.Series('', index=df.index)
    df['_nesb']    = cob.str.contains('non-english', na=False).astype(int) * df['_weight']
    df['_english'] = (cob.str.contains('english', na=False) & ~cob.str.contains('non-english', na=False)).astype(int) * df['_weight']

    # 7. Age bracket flags
    if 'age_group' in df.columns:
        age_norm = df['age_group'].astype(str).str.strip().map(AGE_MAP)
        for col in AGE_COLS:
            df[f'_{col}'] = (age_norm == col).astype(int) * df['_weight']
    else:
        for col in AGE_COLS:
            df[f'_{col}'] = 0

    # 8. Aggregate
    gcols = [c for c in GROUP_COLS if c in df.columns]
    agg_map = {
        'total_users':        ('_weight',     'sum'),
        'n_permanent':        ('_permanent',  'sum'),
        'n_respite':          ('_respite',    'sum'),
        'n_indigenous':       ('_indigenous', 'sum'),
        'n_male':             ('_male',       'sum'),
        'n_female':           ('_female',     'sum'),
        'n_nesb':             ('_nesb',       'sum'),
        'n_english_speaking': ('_english',    'sum'),
    }
    for col in AGE_COLS:
        agg_map[col] = (f'_{col}', 'sum')

    return df.groupby(gcols).agg(**agg_map).reset_index()


results_curf = []
for fname in sorted(os.listdir(RAW_CURF)):
    if fname.startswith('~'): continue
    if not (fname.endswith('.xlsx') or fname.endswith('.csv')): continue

    path = f'{RAW_CURF}/{fname}'
    try:
        df  = load_file(path)
        agg = aggregate_curf_resid(df, fname)
        if agg is not None:
            results_curf.append(agg)
            print(f'OK  {fname}: {df.shape} -> {agg.shape}  years={sorted(agg["year"].unique())}')
        del df
    except Exception as e:
        print(f'ERR {fname}: {e}')

res_users = pd.concat(results_curf, ignore_index=True)
print(f'\nCombined: {res_users.shape}')
print('Years:', sorted(res_users['year'].unique()))
print('ACPRs:', res_users['acpr_code'].nunique())

In [ ]:
# =============================================================================
# STEP 5 — Quick sanity check
# =============================================================================

print(res_users[['acpr_name', 'year', 'total_users',
                  'n_permanent', 'n_respite',
                  'n_indigenous', 'n_male', 'n_female']].head(10).to_string(index=False))
print()
print('Age cols sample:')
print(res_users[['acpr_name', 'year'] + AGE_COLS].head(5).to_string(index=False))

In [ ]:
# =============================================================================
# STEP 5b — Sanity check: compare CURF totals vs GEN snapshot (2023 & 2024)
#
# Snapshot = point-in-time count on 30 June
# CURF     = all users at any point during the year
# Numbers should be in the same ballpark
# =============================================================================

SNAP_FILES_RESID = {
    2023: f'{SNAP_DIR}/GEN-data-People-using-aged-care-by-region-30-June-2023-4-residential-care-(service-location).xlsx',
    2024: f'{SNAP_DIR}/GEN-data-People-using-aged-care-by-region-30-June-2024-4-residential-care-(service-location).xlsx',
}

def load_snapshot_acpr(path):
    raw = pd.read_excel(path, sheet_name='Table 4.3 (ACPR)', header=None, engine='calamine')
    header_row = 0
    for i, row in raw.iterrows():
        if any('permanent' in str(v).lower() or 'total' in str(v).lower() for v in row.values):
            header_row = i
            break
    df = pd.read_excel(path, sheet_name='Table 4.3 (ACPR)', header=header_row, engine='calamine')
    df.columns = [str(c).strip() for c in df.columns]
    code_col = df.columns[0]
    df = df[pd.to_numeric(df[code_col], errors='coerce').notna()].copy()
    return df

rows = []
for yr in [2023, 2024]:
    snap = load_snapshot_acpr(SNAP_FILES_RESID[yr])
    perm_col  = [c for c in snap.columns if 'perm' in c.lower()]
    resp_col  = [c for c in snap.columns if 'resp' in c.lower()]
    total_col = [c for c in snap.columns if c.lower() == 'total']

    snap_perm  = int(snap[perm_col[0]].sum())  if perm_col  else 0
    snap_resp  = int(snap[resp_col[0]].sum())  if resp_col  else 0
    snap_total = int(snap[total_col[0]].sum()) if total_col else snap_perm + snap_resp

    curf_yr    = res_users[res_users['year'] == yr]
    curf_perm  = int(curf_yr['n_permanent'].sum())
    curf_resp  = int(curf_yr['n_respite'].sum())
    curf_total = int(curf_yr['total_users'].sum())

    rows.append({
        'year':        yr,
        'snap_perm':   snap_perm,  'curf_perm':  curf_perm,
        'snap_resp':   snap_resp,  'curf_resp':  curf_resp,
        'snap_total':  snap_total, 'curf_total': curf_total,
        'ratio_perm':  round(curf_perm  / snap_perm,  2) if snap_perm  > 0 else float('nan'),
        'ratio_resp':  round(curf_resp  / snap_resp,  2) if snap_resp  > 0 else float('nan'),
        'ratio_total': round(curf_total / snap_total, 2) if snap_total > 0 else float('nan'),
    })

comp = pd.DataFrame(rows).set_index('year')
print('Snapshot (30 Jun) vs CURF (full-year)\n')
print(comp.to_string())

In [ ]:
# =============================================================================
# STEP 6 — Save
# =============================================================================

res_users.to_csv(OUT_RESID_USERS, index=False)
print(f'Saved: {OUT_RESID_USERS}')
print(f'Shape: {res_users.shape}')